# 00 Bootstrap: Kaggle, Colab or local

Sets up one session for the `afv` pipeline: package install, credentials from a secret store, a cache directory that survives the session, and a runtime copy of the config pointing at that cache.

## What this notebook does and does not show

Does: detect the environment, install `aifs-frost-verif` with the GRIB extras, load `FROST_CLIENT_ID` and `AFV_USER_AGENT` from the secret store or a local `.env`, choose a cache directory, write `config_runtime.yaml`, and print `afv info`.

Does not: download any forecast or observation, and shows no verification result. Run the numbered notebooks 01 to 04 after this one, in the same session or with `AFV_CONFIG` pointing at the config written here.

## Environment

Kaggle mounts `/kaggle`; Colab ships the `google.colab` module; anything else is treated as a local checkout.

In [ ]:
import importlib.util
import os
import sys

if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif importlib.util.find_spec("google.colab") is not None:
    ENV = "colab"
else:
    ENV = "local"
print("environment:", ENV, "| python", sys.version.split()[0])

## Install

The next cell uses the network (GitHub and PyPI). On Kaggle `eccodes` installs cleanly. On Windows it may not; see `docs/DEPLOY.md` for the conda route. Skip this cell in a local checkout where `pip install -e ".[grib]"` has already been run.

In [ ]:
# Network. On Kaggle eccodes installs cleanly; on Windows it may not, see docs/DEPLOY.md.
%pip install -q git+https://github.com/ambrosiobing/aifs-frost-verif ecmwf-opendata cfgrib eccodes

Optional, also network: MET Norway's `verif` for the plots in `03_verify.ipynb`. The pipeline runs without it.

In [ ]:
# Network, optional.
%pip install -q verif

## Credentials

The Frost client id comes only from the Kaggle or Colab secret store, or from a local `.env` read by python-dotenv. Never type it into a cell: the CI workflow fails on any literal id found in a notebook. Add a secret named `FROST_CLIENT_ID` in the Kaggle Add-ons menu or the Colab key panel before running the next cell.

`AFV_USER_AGENT` is the descriptive User-Agent that api.met.no terms of service require; it is read by `afv.fetch_locfc`. Set it as a secret too, or accept the default below.

In [ ]:
import os

if ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["FROST_CLIENT_ID"] = secrets.get_secret("FROST_CLIENT_ID")
    try:
        os.environ["AFV_USER_AGENT"] = secrets.get_secret("AFV_USER_AGENT")
    except Exception:
        pass
elif ENV == "colab":
    from google.colab import userdata
    os.environ["FROST_CLIENT_ID"] = userdata.get("FROST_CLIENT_ID")
    try:
        os.environ["AFV_USER_AGENT"] = userdata.get("AFV_USER_AGENT")
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    load_dotenv()

os.environ.setdefault("AFV_USER_AGENT", "aifs-frost-verif/0.1 github.com/ambrosiobing/aifs-frost-verif")
print("FROST_CLIENT_ID set:", bool(os.environ.get("FROST_CLIENT_ID")))
print("AFV_USER_AGENT:", os.environ["AFV_USER_AGENT"])

## Cache directory

ECMWF open data is kept for about four days, so the GRIB cache has to outlive the session. Kaggle: `/kaggle/working/data/cache` (saved with the notebook output). Colab: a folder on Google Drive; `drive.mount` opens a Google permission prompt in the browser, which you approve yourself. Local: `data/cache` in the checkout.

In [ ]:
from pathlib import Path

if ENV == "kaggle":
    CACHE = Path("/kaggle/working/data/cache")
elif ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/afv_cache")
else:
    CACHE = Path("data/cache")
CACHE.mkdir(parents=True, exist_ok=True)
ROOT = CACHE.parent
print("cache:", CACHE.resolve())
print("root: ", ROOT.resolve())

## Runtime config

The default config lives in `configs/default.yaml`. When the repository is not checked out (Kaggle, Colab after a plain pip install) the next cell downloads that one file from GitHub, which uses the network. All `paths` entries are then rewritten to point under the cache root, and the result is saved as `config_runtime.yaml`. The later notebooks read `AFV_CONFIG` to find it.

In [ ]:
import yaml

src = Path("configs/default.yaml")
if not src.exists():
    # Network: one raw file from the repository.
    import urllib.request
    src = ROOT / "default.yaml"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ambrosiobing/aifs-frost-verif/main/configs/default.yaml", src)

cfg = yaml.safe_load(src.read_text(encoding="utf-8"))
cfg["paths"] = {
    "cache_dir": str(CACHE),
    "fields_dir": str(CACHE / "fields"),
    "obs_dir": str(CACHE / "obs"),
    "stations_csv": str(ROOT / "static" / "stations_tier0.csv"),
    "orography_nc": str(ROOT / "static" / "orog_0p25_norway.nc"),
    "results_dir": str(ROOT / "results" / "tier0"),
}
CONFIG = ROOT / "config_runtime.yaml"
CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
os.environ["AFV_CONFIG"] = str(CONFIG)
print("AFV_CONFIG =", CONFIG)
print(yaml.safe_dump(cfg["paths"], sort_keys=False))

## Package check

`afv info` prints the versions and which optional decoders are present. `cfgrib available` and `eccodes available` are needed for `01_fetch.ipynb` to standardise GRIB; without them the fetch can still download and the NetCDF step has to run elsewhere.

In [ ]:
!afv info